In [27]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

def load_and_prepare_data(file_path):
    data = pd.read_csv(file_path)
    data = data[['Survived','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]
    data['Age'] = data['Age'].fillna(data['Age'].mean())
    data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
    label_encoder = LabelEncoder()
    data['Sex'] = label_encoder.fit_transform(data['Sex'])
    data['Embarked'] = label_encoder.fit_transform(data['Embarked'])
    return data

In [24]:
def train_model(data):

    X = data.drop('Survived', axis=1)
    y = data['Survived']
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"\nModel Accuracy: {accuracy:.2f}")
    return model, X.columns



In [ ]:
def get_user_input():

    print("\nEnter Passenger Details for Survival Prediction")

    pclass = int(input("Passenger Class (1 = First, 2 = Second, 3 = Third): "))
    sex = input("Sex (male/female): ").strip().lower()
    age = float(input("Age: "))
    sibsp = int(input("Number of Siblings/Spouses aboard: "))
    parch = int(input("Number of Parents/Children aboard: "))
    fare = float(input("Ticket Fare: "))
    embarked = input("Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton): ").strip().upper()
    sex_encoded = 1 if sex == "male" else 0

    embark_map = {'C':0, 'Q':1, 'S':2}
    embarked_encoded = embark_map.get(embarked, 2)

    return [[pclass, sex_encoded, age, sibsp, parch, fare, embarked_encoded]]



In [25]:

def predict_survival(model, passenger_data, feature_names):
    passenger_df = pd.DataFrame(passenger_data, columns=feature_names)
    prediction = model.predict(passenger_df)
    if prediction[0] == 1:
        print("\nPrediction Result: Passenger is likely to SURVIVE")
    else:
        print("\nPrediction Result: Passenger is NOT likely to survive")

In [ ]:
def save_model(model, feature_names, model_file="titanic_model.pkl", features_file="feature_names.pkl"):
    with open(model_file, 'wb') as f:
        pickle.dump(model, f)
    with open(features_file, 'wb') as f:
        pickle.dump(feature_names, f)
    print(f"\nModel saved to {model_file}")
    print(f"Feature names saved to {features_file}")

def load_model(model_file="titanic_model.pkl", features_file="feature_names.pkl"):
    with open(model_file, 'rb') as f:
        model = pickle.load(f)
    with open(features_file, 'rb') as f:
        feature_names = pickle.load(f)
    print(f"\nModel loaded from {model_file}")
    return model, feature_names

def main():
    dataset = load_and_prepare_data("dataset/Titanic-Dataset.csv")
    model, feature_names = train_model(dataset)
    save_model(model, feature_names)
    passenger = get_user_input()
    predict_survival(model, passenger, feature_names)

if __name__ == "__main__":
    main()


Model Accuracy: 0.81

Model saved to titanic_model.pkl
Feature names saved to feature_names.pkl

Enter Passenger Details for Survival Prediction

Prediction Result: Passenger is NOT likely to survive


In [29]:
def use_saved_model():
    print("\n" + "="*50)
    print("Using Pre-trained Model from Pickle File")
    print("="*50)
    
    model, feature_names = load_model()
    passenger = get_user_input()
    predict_survival(model, passenger, feature_names)